> **Public (scrubbed) copy:** The manufacturer allow-list uses **placeholder codes** (`MFG_*`) instead of real company names. Pre-map labels in your data or edit the list. Sponsor-specific path hints are generic. **Outputs are cleared.** Modeling: `Predictive_Proj_Code_public.ipynb`.


# Retail panel ETL — data prep (portfolio)

This notebook **ingests retail extracts** (product attributes workbook + yearly POS Excel files), **standardizes keys**, **merges** on UPC, **cleans** the panel (missingness, dates), **collapses** long-tail manufacturers for analysis, and writes **`filtered_manufacturers_data.csv`** — the input used in `Predictive_Proj_Code_public.ipynb`.

**Raw files are not on GitHub** (sponsor / policy). To reproduce the pipeline locally, place the original deliverables in this folder (or in a `Data/` subfolder) with the names referenced in the code.

**Pipeline (high level)**

1. Load attributes + yearly POS → pad UPCs → left join → interim CSV.
2. Drop columns with very high missing rates; profile schema; parse `Time`.
3. Keep a fixed manufacturer list; map others to `Rest_ALL` → export filtered panel.

---


In [ ]:
from pathlib import Path
import os

import pandas as pd

# Optional override:
# export PREDICTIVE_PROJECT_DIR="/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload"
_MANUAL_PROJECT_DIR = Path("/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload")


def resolve_project_dir() -> Path:
    if _MANUAL_PROJECT_DIR is not None:
        return Path(_MANUAL_PROJECT_DIR).expanduser().resolve()
    env = os.environ.get("PREDICTIVE_PROJECT_DIR", "").strip()
    if env:
        return Path(env).expanduser().resolve()

    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "Predictive_proj_public.ipynb").is_file() or (p / "Predictive_proj.ipynb").is_file():
            return p

    fallback_proj = Path("/Users/rukmangadaruku/Library/CloudStorage/OneDrive-TheUniversityofTexasatDallas/Sem 2/proj for git upload")
    if fallback_proj.is_dir():
        return fallback_proj
    return here


PROJECT_DIR = resolve_project_dir()
# Use Data/ when present, otherwise use current project folder.
DATA_DIR = PROJECT_DIR / "Data" if (PROJECT_DIR / "Data").is_dir() else PROJECT_DIR
print(f"PROJECT_DIR={PROJECT_DIR}")
print(f"DATA_DIR={DATA_DIR}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")


In [ ]:
product_attributes = pd.read_excel(DATA_DIR / "Product Attributes.xlsx", sheet_name='Rfg_and_Fz_Meat_and_Substitute')


In [ ]:
yearly_sales_files = [
    DATA_DIR / "Fz_Rfg Substitute Meat_POS_2020.xlsx",
    DATA_DIR / "Fz_Rfg Substitute Meat_POS_2021.xlsx",
    DATA_DIR / "Fz_Rfg Substitute Meat_POS_2022.xlsx",
    DATA_DIR / "Fz_Rfg Substitute Meat_POS_2023.xlsx",
    DATA_DIR / "Fz_Rfg Substitute Meat_POS_2024.xlsx"]

# Step 2: Combine all yearly sales data into a single DataFrame
sales_data = pd.concat([pd.read_excel(file) for file in yearly_sales_files], ignore_index=True)

# Step 3: Data Cleaning
product_attributes['UPC 13 digit'] = product_attributes['UPC 13 digit'].astype(str).str.zfill(13)
sales_data['UPC 13 digit'] = sales_data['UPC 13 digit'].astype(str).str.zfill(13)
# Step 4: Merge product attributes with sales data
merged_data = pd.merge(sales_data, product_attributes, on='UPC 13 digit', how='left')
merged_data.to_csv(DATA_DIR / 'prepared_data.csv', index=False)
print("Data preparation complete. The prepared data is saved as 'prepared_data.csv'.")



In [ ]:
# Step 5.1: Drop columns with more than 90% missing values
threshold = 0.9
missing_ratio = merged_data.isnull().mean()
columns_to_drop = missing_ratio[missing_ratio > threshold].index

merged_data = merged_data.drop(columns=columns_to_drop)

print(f"Dropped columns with >90% missing values: {list(columns_to_drop)}")

In [ ]:
# Count numerical columns (int or float)
numerical_cols = merged_data.select_dtypes(include=['number']).columns
num_numerical = len(numerical_cols)
# Count categorical columns (object or category type)
categorical_cols = merged_data.select_dtypes(include=['object', 'category']).columns
num_categorical = len(categorical_cols)
print(f"Number of numerical columns: {num_numerical}")
print(f"Number of categorical columns: {num_categorical}")

In [ ]:
#print("Numerical columns:", list(numerical_cols))
print("Categorical columns:", list(categorical_cols))

In [ ]:
# Get unique value count for each column
unique_counts = merged_data.nunique().sort_values(ascending=False)

# Display results
print("Number of unique values per column:")
print(unique_counts)

In [ ]:
output_path = DATA_DIR / "prepared_data_cleaned.csv"
merged_data.to_csv(output_path, index=False)

print(f"Overwritten cleaned data saved to: {output_path}")

In [ ]:
# Convert to datetime format (assuming MM-DD-YY)
merged_data['Time'] = merged_data['Time'].str.extract(r'(\d{2}-\d{2}-\d{2})')
merged_data['Time'] = pd.to_datetime(merged_data['Time'], format='%m-%d-%y')
print("'Time' column converted to datetime format.")
print(merged_data['Time'].head())

In [ ]:
# Define the list of manufacturer names to keep
main_manufacturers = [
    'MFG_ALPHA',
    'MFG_BETA',
    'MFG_GAMMA',
    'MFG_DELTA'
]
filtered_data = merged_data.copy()

# Replace all manufacturers not in the main list with 'Rest_ALL'
filtered_data['Manufacturer Name'] = filtered_data['Manufacturer Name'].apply(
    lambda x: x if x in main_manufacturers else 'Rest_ALL'
)

print(f"Grouped data now contains {filtered_data.shape[0]} rows.")
print(filtered_data['Manufacturer Name'].value_counts())

In [ ]:
# Save the filtered data
filtered_path = DATA_DIR / "filtered_manufacturers_data.csv"
filtered_data.to_csv(filtered_path, index=False)

print(f"Filtered data saved to: {filtered_path}")

In [ ]:
# Step 1: Identify categorical columns
categorical_cols = filtered_data.select_dtypes(include=['object', 'category']).columns

# Step 2: Print unique values for each categorical column
for col in categorical_cols:
    unique_vals =filtered_data[col].dropna().unique()
    print(f"\n {col} ({len(unique_vals)} unique values):")
    print(unique_vals)

## Next: modeling & reporting

**LASSO, OLS, merchandising analysis, and related plots** live in **`../Predictive_Proj_Code_public.ipynb`** so this repo stays split: **ETL here, models there.**

Run that notebook after `filtered_manufacturers_data.csv` is produced (paths resolve automatically or via `PREDICTIVE_ROOT` — see parent `README.md`).
